# 230. Kth Smallest Element in a BST
**Difficulty:** 🟡 Medium · **Topic:** Tree · **LeetCode:** https://leetcode.com/problems/kth-smallest-element-in-a-bst/

## 💡 Concepts

**Core concept(s):** An **in-order** walk of a BST produces sorted values — so the k-th one it emits is the answer.

**Why it applies here:** Because in-order visits a BST in increasing order, we just count visits and stop at the k-th. Stopping early avoids walking the whole tree.

**Key intuition:** Walk in sorted (in-order) order and stop the moment you've seen k values.

---

### 📚 What is a Binary Tree?
A **binary tree** is nodes in a branching shape: each node holds a value and up to two children (**left**, **right**). The top is the **root**; childless nodes are **leaves**; **height** is the longest root-to-leaf path.
- **In Python:** a small `TreeNode` class with `.val`, `.left`, `.right`.

### 📚 What is a Binary Search Tree (BST)?
A **BST** stays ordered: for every node, everything in its **left** subtree is smaller and everything in its **right** subtree is larger — so you can find values by going left/right like binary search.
- **Key fact:** an **in-order** walk of a BST visits the values in **sorted** order.

### 📚 What is DFS (Depth-First Search) / Recursion?
**DFS** dives down one branch as far as possible, then backtracks. It's usually written with **recursion** — a function that calls itself on each child.
- **Complexity:** visits each node once → **O(n)** time; uses call-stack space up to the tree's **height**.

---

**Prerequisite knowledge:**
- In-order traversal (recursive or with a stack).

## 📝 Problem

Return the k-th smallest value in a BST (1-indexed).

**Example**
```
[3,1,4,None,2], k=1 -> 1
[5,3,6,2,4,None,None,1], k=3 -> 3
```

> Two approaches: collect all in-order `O(n)` and early-stop in-order `O(h + k)`.

In [ ]:
from typing import Optional, List
from collections import deque

class TreeNode:
    """A single node of a binary tree: a value plus links to up to two children."""
    def __init__(self, val=0, left=None, right=None):
        self.val = val                     # the number stored at this node
        self.left = left                   # the left child (or None)
        self.right = right                 # the right child (or None)

def build_tree(values):
    """Build a tree from a level-order list, LeetCode style (None = missing child)."""
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0]); q = deque([root]); i = 1
    while q and i < len(values):
        node = q.popleft()                 # the parent we're attaching children to
        if i < len(values):                # attach the left child (if present)
            if values[i] is not None:
                node.left = TreeNode(values[i]); q.append(node.left)
            i += 1
        if i < len(values):                # attach the right child (if present)
            if values[i] is not None:
                node.right = TreeNode(values[i]); q.append(node.right)
            i += 1
    return root

def build_balanced(n):
    """Balanced BST holding 1..n (height ~log n) — used by the benchmark."""
    def helper(lo, hi):
        if lo > hi:
            return None
        mid = (lo + hi) // 2               # middle value becomes the subtree's root
        node = TreeNode(mid)
        node.left = helper(lo, mid - 1)    # smaller values go left
        node.right = helper(mid + 1, hi)   # larger values go right
        return node
    return helper(1, n)

def preorder(root):
    """Collect values in preorder: node, then left, then right."""
    out = []
    def go(n):
        if not n: return
        out.append(n.val); go(n.left); go(n.right)
    go(root); return out

def inorder(root):
    """Collect values in inorder: left, then node, then right (sorted for a BST)."""
    out = []
    def go(n):
        if not n: return
        go(n.left); out.append(n.val); go(n.right)
    go(root); return out

def same_shape(a, b):
    """True if two trees have identical shape and values."""
    if not a and not b: return True        # both empty -> match
    if not a or not b or a.val != b.val: return False  # one empty, or values differ
    return same_shape(a.left, b.left) and same_shape(a.right, b.right)

### Approach 1 — Full In-Order List (worst)

**Idea:** Collect every value in sorted order, then index `k-1`.

**Time:** `O(n)`. **Space:** `O(n)`.

In [ ]:
def kth_full(root: Optional[TreeNode], k: int) -> int:
    vals = []

    def ino(n):
        """In-order walk lists values smallest-first."""
        if not n:
            return
        ino(n.left)
        vals.append(n.val)
        ino(n.right)

    ino(root)
    return vals[k - 1]   # k-th smallest (k is 1-indexed)

### Approach 2 — Early-Stop In-Order (optimal)

**Idea:** Walk in-order with a stack, counting; return as soon as the count hits `k` — no need to visit the rest.

**Time:** `O(h + k)`.

**Space:** `O(h)`.

In [ ]:
def kth_early(root: Optional[TreeNode], k: int) -> int:
    stack, node, count = [], root, 0
    while stack or node:
        while node:                        # dive to the smallest unvisited node
            stack.append(node)
            node = node.left
        node = stack.pop()
        count += 1                         # we've now visited one more value (in sorted order)
        if count == k:                     # reached the k-th smallest...
            return node.val                # ...return it immediately (no need to continue)
        node = node.right
    return -1

In [ ]:
# Correctness check
tests = [([3,1,4,None,2],1,1), ([5,3,6,2,4,None,None,1],3,3), ([1],1,1)]
for vals, k, exp in tests:
    root = build_tree(vals)
    a, b = kth_full(root, k), kth_early(root, k)
    print(f"{vals}, k={k} -> full={a}, early={b} | expected={exp}")
    assert a == b == exp, "mismatch!"
print("\nAll tests passed")

## ⏱️ Empirically Checking the Complexities

Big-O can't be read off a function directly, but it can be **measured**. We time each approach on trees of growing size `n` and read the **doubling ratio** — how much runtime grows when `n` doubles.

| Theoretical | Ratio when `n` → `2n` |
|-------------|-----------------------|
| `O(log n)`    | ≈ **1×** |
| `O(n)`        | ≈ **2×** |
| `O(n²)`       | ≈ **4×** |

We use **balanced** trees (height ~log n) so deep recursion stays safe while every node is still visited.

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark

def make_worst_case(n):
    return (build_balanced(n), 1)   # k=1 -> early-stop finishes almost immediately
solutions = {
    "full  O(n)    ": kth_full,
    "early O(h + k)": kth_early,
}
sizes = [1000, 2000, 4000, 8000]

benchmark(solutions, make_worst_case, sizes, plot=True)


## 🧩 Patterns Learned

- **In-order for order statistics:** the k-th smallest/largest in a BST falls out of an in-order walk.
- **Stop early:** returning as soon as you have the answer beats collecting everything.
- **Signal:** "k-th smallest/largest", "median of a BST".
- **Related problems:** Validate BST, BST Iterator, Kth Largest Element.
- **Common pitfalls:** (1) 1-indexed vs 0-indexed k; (2) collecting the whole tree when early exit would do.